# Notebook 01 — O boom nas buscas
**Déficit de Dado · ed002**

Série histórica de interesse por GLP-1 no Google Brasil (2019–2025).
Fonte: Google Trends via pytrends.

Para rodar no seu ambiente:
```
conda activate midia
pip install pytrends
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from pytrends.request import TrendReq
import time

In [ ]:
# --- COLETA VIA PYTRENDS ---
# Roda isso no seu ambiente local. A API do Google bloqueia em servidores.

pytrends = TrendReq(hl='pt-BR', tz=-180)  # tz=-180 = horário de Brasília

termos = ['ozempic', 'semaglutida', 'canetas emagrecedoras', 'dieta para emagrecer']

# Google Trends aceita no máximo 5 termos por chamada
pytrends.build_payload(
    termos,
    cat=0,
    timeframe='2019-01-01 2025-12-31',
    geo='BR'
)
time.sleep(2)  # respeita rate limit

df_raw = pytrends.interest_over_time()

# remove coluna isPartial se existir
if 'isPartial' in df_raw.columns:
    df_raw = df_raw.drop(columns=['isPartial'])

df_raw.index = pd.to_datetime(df_raw.index)
df_raw.to_csv('trends_raw.csv')  # salva pra não ter que rodar de novo
print(df_raw.tail())

In [ ]:
# --- SE JÁ RODOU ANTES, CARREGA DO CSV ---
# df_raw = pd.read_csv('trends_raw.csv', index_col=0, parse_dates=True)

In [ ]:
# --- DADOS DE REFERÊNCIA (coletados manualmente para reprodução) ---
# Fonte: Conexa Saúde / Medicina S/A (agosto 2025), Google Trends análise pública
# Usados como anotações no gráfico — não substituem a série do pytrends

marcos = {
    '2021-01': 'Primeiros relatos\noff-label nos EUA',
    '2022-07': 'Escassez global\ndo Ozempic',
    '2023-01': 'Buscas sobem 91%\nno 1º sem/2023',
    '2023-07': 'TikTok acelera\nadoção no BR',
    '2025-08': '"Canetas emagrecedoras"\nultrapassou "dietas"',
}

In [ ]:
# --- VISUALIZAÇÃO ---

fig, ax = plt.subplots(figsize=(14, 6))

cores = {
    'ozempic': '#C0392B',
    'semaglutida': '#E67E22',
    'canetas emagrecedoras': '#8E44AD',
    'dieta para emagrecer': '#2980B9',
}

for termo, cor in cores.items():
    if termo in df_raw.columns:
        ax.plot(
            df_raw.index,
            df_raw[termo],
            color=cor,
            linewidth=2,
            label=termo.title(),
            alpha=0.9
        )

# anotações dos marcos
for data_str, texto in marcos.items():
    data = pd.Timestamp(data_str)
    if df_raw.index.min() <= data <= df_raw.index.max():
        ax.axvline(x=data, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
        ax.text(
            data, 102, texto,
            fontsize=7.5, ha='center', va='bottom',
            color='#555555',
            rotation=0,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.8)
        )

ax.set_xlabel('')
ax.set_ylabel('Interesse relativo (Google Trends)', fontsize=10)
ax.set_title(
    'Interesse por GLP-1 no Google Brasil — 2019 a 2025',
    fontsize=13, fontweight='bold', pad=16
)
ax.set_ylim(0, 115)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%d'))
ax.legend(loc='upper left', framealpha=0.9, fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

# nota de fonte
fig.text(
    0.01, -0.02,
    'Fonte: Google Trends (pytrends) · geo=BR · escala relativa ao pico histórico de cada série',
    fontsize=8, color='gray'
)

plt.tight_layout()
plt.savefig('nb01_boom_buscas.png', dpi=180, bbox_inches='tight')
plt.show()
print("Gráfico salvo: nb01_boom_buscas.png")

In [ ]:
# --- GRÁFICO 2: comparação canetas vs dieta no pico (2025) ---
# Mostra o momento em que "canetas emagrecedoras" ultrapassou "dietas"

df_2025 = df_raw[df_raw.index >= '2025-01-01'].copy()

if not df_2025.empty and 'canetas emagrecedoras' in df_2025.columns:
    fig2, ax2 = plt.subplots(figsize=(10, 5))

    ax2.plot(df_2025.index, df_2025['canetas emagrecedoras'],
             color='#8E44AD', linewidth=2.5, label='canetas emagrecedoras')
    ax2.plot(df_2025.index, df_2025['dieta para emagrecer'],
             color='#2980B9', linewidth=2.5, label='dieta para emagrecer', linestyle='--')

    # marca o cruzamento (agosto 2025 segundo dados públicos)
    cruzamento = pd.Timestamp('2025-08-01')
    ax2.axvline(x=cruzamento, color='#E74C3C', linestyle=':', linewidth=1.5)
    ax2.text(cruzamento, 95, 'aqui\ncanetas > dietas', fontsize=8,
             ha='center', color='#E74C3C',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none'))

    ax2.set_title('Quando o remédio venceu a dieta nas buscas', fontsize=12, fontweight='bold', pad=14)
    ax2.set_ylabel('Interesse relativo', fontsize=10)
    ax2.set_ylim(0, 110)
    ax2.legend(fontsize=9, framealpha=0.9)
    ax2.spines[['top', 'right']].set_visible(False)
    fig2.text(0.01, -0.02, 'Fonte: Google Trends (pytrends) · geo=BR · 2025', fontsize=8, color='gray')
    plt.tight_layout()
    plt.savefig('nb01_canetas_vs_dieta.png', dpi=180, bbox_inches='tight')
    plt.show()
    print("Gráfico salvo: nb01_canetas_vs_dieta.png")